# 01. Exploratory data analysis

Start by exploring the dataset and get a feel for it.

In [1]:
# Set up the connection to the database
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/bank.db")

### Column names and basic statistics

In [2]:
# Get a feel for the data (average age, call duration, ...)
df = pd.read_sql("""SELECT * FROM clients""", conn)
df.describe()

,row_id,age,duration,campaign,pdays,previous,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,subscribed
count,41188.00000,41188.00000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000
mean,20593.50000,40.02406,258.285010,2.567593,962.475454,0.172963,0.081886,93.575664,-40.502600,3.621291,5167.035911,0.112654
std,11890.09578,10.42125,259.279249,2.770014,186.910907,0.494901,1.570960,0.578840,4.628198,1.734447,72.251528,0.316173
min,0.00000,17.00000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.634000,4963.600000,0.000000
25%,10296.75000,32.00000,102.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.344000,5099.100000,0.000000
50%,20593.50000,38.00000,180.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000,0.000000
75%,30890.25000,47.00000,319.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000,0.000000
max,41187.00000,98.00000,4918.000000,56.000000,999.000000,7.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000,1.000000


### Find out the overall subscription rate

In [3]:
# Check the subscription rate
query = """
    SELECT COUNT(*) AS n_clients,
        ROUND(100.0 * AVG(subscribed), 2) AS pct_subscribed
    FROM clients;
"""
pd.read_sql(query, conn)

,n_clients,pct_subscribed
0,41188,11.27


### Find out the subscription rate per job

In [4]:
# Conversion rate by job
pd.read_sql("""SELECT job, COUNT(*) AS n_clients, AVG(100.0 * subscribed) AS pct_subscribed
    FROM clients GROUP BY job
    ORDER BY pct_subscribed DESC""", conn)

,job,n_clients,pct_subscribed
0,student,875,31.428571
1,retired,1720,25.232558
2,unemployed,1014,14.201183
3,admin.,10422,12.972558
4,management,2924,11.217510
5,unknown,330,11.212121
6,technician,6743,10.826042
7,self-employed,1421,10.485574
8,housemaid,1060,10.000000
9,entrepreneur,1456,8.516484


### Find out the subscription rate per age band

We do ten-year intervals

In [5]:
# Conversion rate by age band
pd.read_sql("""
    SELECT CASE
        WHEN age < 25 THEN '<25'
        WHEN age < 35 THEN '25-34'
        WHEN age < 45 THEN '35-44'
        WHEN age < 55 THEN '45-54'
        WHEN age < 65 THEN '55-64'
        ELSE '65+'
    END AS age_band,
    COUNT(*) AS n_clients,
    AVG(100.0 * subscribed) AS pct_subscribed
    FROM clients
    GROUP BY age_band
    ORDER BY MIN(age)""",
    conn
)

,age_band,n_clients,pct_subscribed
0,<25,1068,23.970037
1,25-34,13686,12.173024
2,35-44,13500,8.651852
3,45-54,8704,8.651195
4,55-64,3567,13.568825
5,65+,663,47.209653


### Find out the subscription rate per month

In [6]:
# Conversion rate by month
pd.read_sql("""
    SELECT month, COUNT(*) as n_clients, AVG(100.0 * subscribed) as pct_subscribed
    FROM clients
    GROUP BY month
    ORDER BY CASE month
        WHEN 'jan' THEN 1 WHEN 'feb' THEN 2 WHEN 'mar' THEN 3
        WHEN 'apr' THEN 4 WHEN 'may' THEN 5 WHEN 'jun' THEN 6
        WHEN 'jul' THEN 7 WHEN 'aug' THEN 8 WHEN 'sep' THEN 9
        WHEN 'oct' THEN 10 WHEN 'nov' THEN 11 WHEN 'dec' THEN 12
    END;

    """,
   conn
)

,month,n_clients,pct_subscribed
0,mar,546,50.549451
1,apr,2632,20.478723
2,may,13769,6.434745
3,jun,5318,10.511470
4,jul,7174,9.046557
5,aug,6178,10.602137
6,sep,570,44.912281
7,oct,718,43.871866
8,nov,4101,10.143867
9,dec,182,48.901099


### Find out the years

This is not a column available in the original dataset, so we make one.

We will also add a similar routine to src/ as this will be used often.

In [7]:
# Recover the years (have same months in different years, but we can use that the dataset is ordered chronologically)
# Skip january and february as there are no subscribers in those months
months = {"mar": 3, "apr": 4, "may": 5, "jun": 6, "jul": 7,
          "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}
month_num = df["month"].map(months)

new_year = (month_num < month_num.shift())  # True where the month drops vs. the previous row
df["year_block"] = new_year.cumsum()        # Running sum to assign each row a specific year starting from 0

Now we can group by year and do some basic operations. Let's first check that the blocks come out as expected.

In [8]:
# Check that the year blocks have been computed correctly
df['month_num'] = month_num                        # Store month_num as a new column in df
print(month_num.isna().sum())                      # 1. Check that there's no missing month numbers (should be 0)
print(new_year.sum(), df['year_block'].nunique())  # 2. Check that there's one less jump than blocks, expect 2 jumps and 3 blocks
print(df.groupby('year_block').size().sum())       # 3. Check that all rows were mapped, expect 41188
print(df.groupby('year_block')['month_num'].apply(lambda s: s.is_monotonic_increasing))  # 4. Check that the months only increase, expect all True

0
2 3
41188
year_block
0    True
1    True
2    True
Name: month_num, dtype: bool


### Some additional checks

Share of clients not contacted before, as well as unknown data.

We find that the same clients that have unknown housing also have unknown loan.

There's 12 duplicate rows.

In [9]:
print(f"Subscribed: {100 * df['subscribed'].mean():.2f}%")  # overall rate
cols = df.columns.drop(["row_id", "year_block", "month_num"])
print(f"Duplicate rows: {df.duplicated(subset=cols).sum()}")  # duplicate rows
print(f"Never contacted before: {100 * (df['pdays'] == 999).mean():.2f}%")  # share "never contacted before"
print('"Unknown" per column:')
print((df == "unknown").sum())  # "unknown" per column
print(((df["housing"] == "unknown") & (df["loan"] == "unknown")).sum())
print(df["default"].value_counts())

Subscribed: 11.27%
Duplicate rows: 12
Never contacted before: 96.32%
"Unknown" per column:
row_id               0
age                  0
job                330
marital             80
education         1731
default           8597
housing            990
loan               990
contact              0
month                0
day_of_week          0
duration             0
campaign             0
pdays                0
previous             0
poutcome             0
emp_var_rate         0
cons_price_idx       0
cons_conf_idx        0
euribor3m            0
nr_employed          0
subscribed           0
year_block           0
month_num            0
dtype: int64
990
default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64


### Year-resolved average euribor and subscription rate

We find that the subscription rate rose dramatically towards the last year.

At the same time, the euribor dropped significantly.

In [10]:
# Get the number of clients, average euribor, and subscription rate for each year
summary = df.groupby('year_block').agg(
    first_month=('month', 'first'),
    last_month=('month', 'last'),
    n_clients=('month', 'count'),
    avg_euribor=('euribor3m', 'mean'),
    pct_subscribed=('subscribed', 'mean'),
)
summary['pct_subscribed'] = summary['pct_subscribed'] * 100
summary.round(2)

,first_month,last_month,n_clients,avg_euribor,pct_subscribed
year_block,,,,,
0,may,dec,27690,4.82,4.84
1,mar,dec,11440,1.23,19.48
2,mar,nov,2058,0.81,52.14


### Split the training and test data

We now use the load_data and add_year_block helper functions inside dataset.py

The subscription rate of the training data is significantly lower than that of the test data.

In [11]:
import sys
sys.path.append("..")

from src.dataset import load_clients, add_year_block, time_split

df = add_year_block(load_clients())
train, test = time_split(df)

In [12]:
# Last 20% will be the test data
cut = int(len(df) * 0.8)    # 32950
train = df.iloc[:cut]
test = df.iloc[cut:]

In [13]:
print(len(train), len(test))
print(train['subscribed'].mean(), test['subscribed'].mean())
print(test['year_block'].value_counts())
print(test['month'].value_counts())

32940 8236
0.06375227686703097 0.3082807187955318
year_block
1    6178
2    2058
Name: count, dtype: int64
month
may    3486
aug    1003
jun     944
oct     650
sep     570
jul     488
nov     485
mar     264
apr     174
dec     172
Name: count, dtype: int64


Explore which months result in more subscribers, both in the full set as well as in the splits. 

In [14]:
pd.crosstab(df["year_block"], df["month"])

month,apr,aug,dec,jul,jun,mar,may,nov,oct,sep
year_block,,,,,,,,,,
0,0,5173,10,6681,4374,0,7762,3615,67,0
1,2457,770,172,177,715,282,5793,357,446,267
2,174,233,0,311,229,264,212,128,204,303


In [15]:
 pd.crosstab(train["year_block"], train["month"])

month,apr,aug,dec,jul,jun,mar,may,nov,oct
year_block,,,,,,,,,
0,0,5173,10,6681,4374,0,7762,3615,67
1,2457,0,0,0,0,282,2519,0,0


In [16]:
 pd.crosstab(test["year_block"], test["month"])

month,apr,aug,dec,jul,jun,mar,may,nov,oct,sep
year_block,,,,,,,,,,
1,0,770,172,177,715,0,3274,357,446,267
2,174,233,0,311,229,264,212,128,204,303
